# LangChain Agent State: Independent CRUD Examples

Each CRUD section is **standalone**. It has its own imports, Pydantic model, state schema, tool, agent, checkpointer, setup state, and invocation.

You can run **any one section independently** from its first code cell. It does not depend on another CRUD section.

**Scope:** Short-term agent state only, using `ToolRuntime.state` to read state and `Command(update=...)` to update it, as described in the [official LangChain Tools documentation](https://docs.langchain.com/oss/python/langchain/tools#access-context).

> Before running a section, set `OPENAI_API_KEY` in your environment or in a `.env` file. Each section uses `InMemorySaver` and its own thread ID, so its demo state is isolated.

# 1. CREATE — Create a user profile

### Objective
Create a profile if one does not already exist.

### How it works
1. `runtime.state.get("user_profile")` checks the current state.
2. `UserProfile(...)` validates the supplied values.
3. `model_dump()` converts the Pydantic object into a dictionary for state storage.
4. `Command(update=...)` updates `user_profile` and adds a matching `ToolMessage`.

### Independent execution
The code below defines everything needed for this CREATE example. It starts with `user_profile=None`; it does not rely on any other notebook section.

In [2]:
# CREATE — standalone implementation

from typing import Any

from pydantic import BaseModel, Field

from langchain.agents import AgentState, create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


# 1. Pydantic model
class CreateUserProfile(BaseModel):
    name: str
    age: int
    role: str
    skills: list[str] = Field(default_factory=list)


# 2. Agent state schema
class CreateState(AgentState):
    user_profile: dict[str, Any] | None


# 3. CREATE tool
@tool
def create_profile(
    name: str,
    age: int,
    role: str,
    runtime: ToolRuntime[None, CreateState],
) -> Command:
    """Create a user profile in the current conversation state."""

    if runtime.state.get("user_profile") is not None:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="A user profile already exists.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    profile = CreateUserProfile(name=name, age=age, role=role)

    return Command(
        update={
            "user_profile": profile.model_dump(),
            "messages": [
                ToolMessage(
                    content="User profile created successfully.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# 4. Independent agent + checkpointer
create_checkpointer = InMemorySaver()

create_agent_app = create_agent(
    model="openai:gpt-5.5",
    tools=[create_profile],
    state_schema=CreateState,
    checkpointer=create_checkpointer,
    system_prompt=(
        "You manage user profiles. Use create_profile when the user asks "
        "to create a profile. Do not claim success unless the tool confirms it."
    ),
)

# 5. Invoke CREATE with a fresh, independent thread
create_config = {"configurable": {"thread_id": "crud-create-demo"}}

create_result = create_agent_app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Create my profile. My name is Pratik, age is 26, "
                    "and my role is Senior Gen AI Engineer."
                ),
            }
        ],
        "user_profile": None,
    },
    config=create_config,
)

print("CREATE result:")
print(create_result["user_profile"])

CREATE result:
{'name': 'Pratik', 'age': 26, 'role': 'Senior Gen AI Engineer', 'skills': []}


# 2. READ — Retrieve a user profile

### Objective
Read the existing profile without changing the state.

### How it works
1. `runtime.state.get("user_profile")` reads the stored value.
2. `UserProfile.model_validate(...)` validates the stored dictionary.
3. `model_dump_json()` returns a readable JSON representation.
4. This tool returns a string; it does not need `Command` because it does not mutate state.

### Independent execution
This section seeds its own sample profile when it starts. It does not require the CREATE section to have run.

In [3]:
# READ — standalone implementation


from typing import Any

from pydantic import BaseModel, Field

from langchain.agents import AgentState, create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver



# 1. Pydantic model
class ReadUserProfile(BaseModel):
    name: str
    age: int
    role: str
    skills: list[str] = Field(default_factory=list)


# 2. Agent state schema
class ReadState(AgentState):
    user_profile: dict[str, Any] | None


# 3. READ tool
@tool
def read_profile(runtime: ToolRuntime[None, ReadState]) -> str:
    """Read the user profile from the current conversation state."""

    profile_data = runtime.state.get("user_profile")

    if profile_data is None:
        return "No user profile found."

    profile = ReadUserProfile.model_validate(profile_data)
    return profile.model_dump_json(indent=2)


# 4. Independent agent + checkpointer
read_checkpointer = InMemorySaver()

read_agent_app = create_agent(
    model="openai:gpt-5.5",
    tools=[read_profile],
    state_schema=ReadState,
    checkpointer=read_checkpointer,
    system_prompt=(
        "You manage user profiles. Use read_profile when the user asks "
        "to view or retrieve their profile. Report the tool result accurately."
    ),
)

# 5. Seed this section's own state and invoke READ
read_config = {"configurable": {"thread_id": "crud-read-demo"}}

read_initial_profile = ReadUserProfile(
    name="Pratik",
    age=26,
    role="Senior Gen AI Engineer",
    skills=["Python", "LangChain"],
).model_dump()

read_result = read_agent_app.invoke(
    {
        "messages": [
            {"role": "user", "content": "Show me my user profile."}
        ],
        "user_profile": read_initial_profile,
    },
    config=read_config,
)

print("READ result:")
print(read_result["user_profile"])

READ result:
{'name': 'Pratik', 'age': 26, 'role': 'Senior Gen AI Engineer', 'skills': ['Python', 'LangChain']}


# 3. UPDATE — Modify an existing user profile

### Objective
Update only the fields supplied by the user and preserve the other values.

### How it works
1. Read the existing dictionary from `runtime.state`.
2. Validate it by creating a `UserProfile` instance.
3. Collect the supplied non-`None` values.
4. Merge those values with the existing profile.
5. Validate the merged data again with Pydantic.
6. Return `Command(update=...)` with the updated dictionary and a matching `ToolMessage`.

### Independent execution
This section creates its own initial profile in its own thread. It does not depend on CREATE or READ.

In [4]:
# UPDATE — standalone implementation

from typing import Any

from pydantic import BaseModel, Field

from langchain.agents import AgentState, create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


# 1. Pydantic model
class UpdateUserProfile(BaseModel):
    name: str
    age: int
    role: str
    skills: list[str] = Field(default_factory=list)


# 2. Agent state schema
class UpdateState(AgentState):
    user_profile: dict[str, Any] | None


# 3. UPDATE tool
@tool
def update_profile(
    runtime: ToolRuntime[None, UpdateState],
    name: str | None = None,
    age: int | None = None,
    role: str | None = None,
    skills: list[str] | None = None,
) -> Command:
    """Update only the supplied fields of the existing user profile."""

    profile_data = runtime.state.get("user_profile")

    if profile_data is None:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="No profile exists to update. Create one first.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    current_profile = UpdateUserProfile.model_validate(profile_data)

    changes = {
        "name": name,
        "age": age,
        "role": role,
        "skills": skills,
    }
    changes = {key: value for key, value in changes.items() if value is not None}

    updated_profile = UpdateUserProfile.model_validate(
        {**current_profile.model_dump(), **changes}
    )

    return Command(
        update={
            "user_profile": updated_profile.model_dump(),
            "messages": [
                ToolMessage(
                    content="User profile updated successfully.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# 4. Independent agent + checkpointer
update_checkpointer = InMemorySaver()

update_agent_app = create_agent(
    model="openai:gpt-5.5",
    tools=[update_profile],
    state_schema=UpdateState,
    checkpointer=update_checkpointer,
    system_prompt=(
        "You manage user profiles. Use update_profile when the user asks "
        "to modify a profile. Only change the fields requested by the user."
    ),
)

# 5. Seed this section's own profile and invoke UPDATE
update_config = {"configurable": {"thread_id": "crud-update-demo"}}

update_initial_profile = UpdateUserProfile(
    name="Pratik",
    age=26,
    role="Senior Gen AI Engineer",
    skills=["Python"],
).model_dump()

update_result = update_agent_app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Change my role to AI Architect and add "
                    "LangChain and FastAPI to my skills."
                ),
            }
        ],
        "user_profile": update_initial_profile,
    },
    config=update_config,
)

print("UPDATE result:")
print(update_result["user_profile"])

UPDATE result:
{'name': 'Pratik', 'age': 26, 'role': 'AI Architect', 'skills': ['LangChain', 'FastAPI']}


# 4. DELETE — Delete a user profile

### Objective
Clear the profile value from the active agent state.

### How it works
1. Check whether `user_profile` exists.
2. If it is already `None`, report that there is nothing to delete.
3. Otherwise, return `Command(update=...)` with `user_profile: None`.
4. Include a matching `ToolMessage` so the model receives the tool result.

### Independent execution
This section seeds its own sample profile in its own thread. It does not depend on CREATE, READ, or UPDATE.

> This example performs a logical deletion by setting the state field to `None`. The state key remains present; this is not physical removal of the key.

In [5]:
# DELETE — standalone implementation

from typing import Any

from pydantic import BaseModel, Field

from langchain.agents import AgentState, create_agent
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


# 1. Pydantic model
class DeleteUserProfile(BaseModel):
    name: str
    age: int
    role: str
    skills: list[str] = Field(default_factory=list)


# 2. Agent state schema
class DeleteState(AgentState):
    user_profile: dict[str, Any] | None


# 3. DELETE tool
@tool
def delete_profile(runtime: ToolRuntime[None, DeleteState]) -> Command:
    """Delete the user profile from the current conversation state."""

    if runtime.state.get("user_profile") is None:
        return Command(
            update={
                "messages": [
                    ToolMessage(
                        content="No profile exists to delete.",
                        tool_call_id=runtime.tool_call_id,
                    )
                ]
            }
        )

    return Command(
        update={
            "user_profile": None,
            "messages": [
                ToolMessage(
                    content="User profile deleted successfully.",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )


# 4. Independent agent + checkpointer
delete_checkpointer = InMemorySaver()

delete_agent_app = create_agent(
    model="openai:gpt-5.5",
    tools=[delete_profile],
    state_schema=DeleteState,
    checkpointer=delete_checkpointer,
    system_prompt=(
        "You manage user profiles. Use delete_profile when the user asks "
        "to delete their profile. Do not claim success unless the tool confirms it."
    ),
)

# 5. Seed this section's own profile and invoke DELETE
delete_config = {"configurable": {"thread_id": "crud-delete-demo"}}

delete_initial_profile = DeleteUserProfile(
    name="Pratik",
    age=26,
    role="Senior Gen AI Engineer",
    skills=["Python", "LangChain"],
).model_dump()

delete_result = delete_agent_app.invoke(
    {
        "messages": [
            {"role": "user", "content": "Delete my user profile."}
        ],
        "user_profile": delete_initial_profile,
    },
    config=delete_config,
)

print("DELETE result:")
print(delete_result["user_profile"])

DELETE result:
None


## Summary

| CRUD operation | State interaction |
|---|---|
| CREATE | Checks state, validates a new Pydantic model, then writes with `Command` |
| READ | Reads through `runtime.state` and returns data without changing state |
| UPDATE | Reads, validates, merges requested changes, then writes with `Command` |
| DELETE | Sets the profile field to `None` with `Command` |

### Core concepts

- **Pydantic model:** validates the profile's data.
- **Agent state schema:** declares the custom state field.
- **`runtime.state`:** gives a tool read access to the current state.
- **`Command(update=...)`:** returns state changes for LangGraph to apply.
- **`ToolMessage`:** reports the tool result to the model and must use the matching `runtime.tool_call_id`.
- **Checkpointer + `thread_id`:** lets state persist across invocations for that conversation.

Each CRUD demo intentionally has its own agent, checkpointer, and thread ID so the sections can be run independently.